In [8]:
!pip install transformers datasets seqeval accelerate

Define Labels and Load CoNLL Format


In [9]:
import os

# Check if uploaded
print("Files in working directory:", os.listdir())


Files in working directory: ['.config', 'drive', 'logs', 'conll_dataset.txt', 'results', 'sample_data']


In [10]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# Define all possible labels
label_list = ['O', 'I-O', 'B-Product', 'I-Product', 'B-PRICE', 'I-PRICE', 'B-LOC', 'I-LOC', 'B-CONTACT_INFO', '@Leyueqa']
label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for label, i in label_to_id.items()}

In [11]:
# Read CoNLL file
def read_conll(path):
    sentences = []
    labels = []
    with open(path, 'r', encoding='utf-8') as f:
        words = []
        tags = []
        for line in f:
            if line.strip() == "":
                if words:
                    sentences.append(words)
                    labels.append(tags)
                    words = []
                    tags = []
            else:
                token, tag = line.strip().split()
                words.append(token)
                tags.append(tag)
        # In case file does not end with a blank line
        if words:
            sentences.append(words)
            labels.append(tags)
    return sentences, labels

In [12]:
# Load your data
sentences, tags = read_conll("conll_dataset.txt")

# Map tag names to IDs
encoded_tags = [[label_to_id[tag] for tag in seq] for seq in tags]

# Create a HuggingFace-compatible dataset
dataset = Dataset.from_dict({"tokens": sentences, "ner_tags": encoded_tags})

In [13]:
train_test = dataset.train_test_split(test_size=0.2, seed=42)
dataset = DatasetDict({
    "train": train_test["train"],
    "validation": train_test["test"]
})



In [14]:
dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 24
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 6
    })
})

 Tokenize the Data and Align Labels

In [15]:
from transformers import AutoTokenizer

# Choose your base model (you can switch to another later for comparison)
model_checkpoint = "xlm-roberta-base"  # You can also try 'Davlan/bert-base-amharic', 'Davlan/afro-xlmr-base'

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Align word-level labels with tokenized subwords
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], is_split_into_words=True, truncation=True, padding='max_length', max_length=128)
    labels = []
    for i, input_ids in enumerate(tokenized_inputs["input_ids"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        current_labels = []
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                current_labels.append(-100)  # special tokens
            elif word_idx != previous_word_idx:
                current_labels.append(examples["ner_tags"][i][word_idx])
            else:
                current_labels.append(examples["ner_tags"][i][word_idx] if label_list[examples["ner_tags"][i][word_idx]].startswith("I") else -100)
            previous_word_idx = word_idx
        labels.append(current_labels)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply tokenization to the entire dataset
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

 Load the Model and Define Training Parameters

In [16]:
!pip install evaluate

In [17]:
from transformers import AutoModelForTokenClassification, Trainer
from transformers.training_args import TrainingArguments
import numpy as np
from datasets import load_metric
import evaluate

# Load pre-trained model for token classification
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id
)

# Define evaluation metric
metric = evaluate.load("seqeval")

# Compute metrics function
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id_to_label[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    true_labels = [
        [id_to_label[l] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]

    return metric.compute(predictions=true_predictions, references=true_labels)

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Define TrainingArguments and Start Fine-Tuning

In [18]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_overall_f1",
    report_to="none" # Disable wandb logging
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Start training
trainer.train()

/tmp/ipython-input-18-1929508207.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Contact Info,Loc,Price,Product,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
1,No log,2.324587,"{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 14}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 5}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 8}",0.000000,0.000000,0.000000,0.165266
2,No log,2.084072,"{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 14}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 5}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 8}",0.000000,0.000000,0.000000,0.229692
3,No log,1.840199,"{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 14}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 5}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 8}",0.000000,0.000000,0.000000,0.526611
4,2.176300,1.659533,"{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 14}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 5}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 8}",0.000000,0.000000,0.000000,0.588235
5,2.176300,1.571259,"{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 14}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 5}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 8}",0.000000,0.000000,0.000000,0.593838


/usr/local/lib/python3.11/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-sco

TrainOutput(global_step=15, training_loss=2.010782559712728, metrics={'train_runtime': 747.1351, 'train_samples_per_second': 0.161, 'train_steps_per_second': 0.02, 'total_flos': 7839469670400.0, 'train_loss': 2.010782559712728, 'epoch': 5.0})

Save the fine-tuned model



In [19]:
model.save_pretrained("amharic-ner-model")
tokenizer.save_pretrained("amharic-ner-model")

('amharic-ner-model/tokenizer_config.json',
 'amharic-ner-model/special_tokens_map.json',
 'amharic-ner-model/sentencepiece.bpe.model',
 'amharic-ner-model/added_tokens.json',
 'amharic-ner-model/tokenizer.json')

Evalute the model on the evalution set


In [21]:
metrics = trainer.evaluate()
print("Validation Performance:")
for k, v in metrics.items():
    if isinstance(v, dict):
        print(f"{k}:")
        for metric_name, value in v.items():
            print(f"  {metric_name}: {value:.4f}")
    else:
        print(f"{k}: {v:.4f}")

Validation Performance:
eval_loss: 2.3246
eval_CONTACT_INFO:
  precision: 0.0000
  recall: 0.0000
  f1: 0.0000
  number: 14.0000
eval_LOC:
  precision: 0.0000
  recall: 0.0000
  f1: 0.0000
  number: 5.0000
eval_PRICE:
  precision: 0.0000
  recall: 0.0000
  f1: 0.0000
  number: 2.0000
eval_Product:
  precision: 0.0000
  recall: 0.0000
  f1: 0.0000
  number: 8.0000
eval_overall_precision: 0.0000
eval_overall_recall: 0.0000
eval_overall_f1: 0.0000
eval_overall_accuracy: 0.1653
eval_runtime: 2.2568
eval_samples_per_second: 2.6590
eval_steps_per_second: 0.4430
epoch: 5.0000


/usr/local/lib/python3.11/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
